# 04 · Modelado
### Fase 4 de CRISP-DM — OE2, OE3

**Objetivo de este notebook:** entrenar y comparar los tres modelos de clasificación binaria
planteados en el anteproyecto — Regresión Logística (baseline), Random Forest y XGBoost — para
predecir `Alto_Riesgo` a partir del dataset construido en `03_Preparacion_de_los_Datos.ipynb`.

**Punto de partida importante:** el split temporal (`Periodo`) y la variable objetivo
(`Alto_Riesgo`) ya vienen resueltos desde el Notebook 3, con el umbral calculado **solo con datos
de entrenamiento (2018–2022)** para evitar fuga de información hacia el período de prueba
(2023–2024). Este notebook reutiliza ese split — no genera uno nuevo.

Estructura:
1. Introducción
2. Importación de librerías
3. Carga del dataset procesado
4. Definición de variables X e y
5. Reutilización del split temporal (`Periodo`)
6. Preprocesamiento
7. Modelo 1: Regresión Logística
8. Modelo 2: Random Forest
9. Modelo 3: XGBoost
10. Validación cruzada
11. Comparación preliminar de modelos
12. Guardado de modelos

> La evaluación exhaustiva (matriz de confusión, curvas ROC, importancia de variables) se hace en `05_Evaluacion.ipynb` — aquí solo se deja una comparación preliminar para elegir con qué modelo(s) avanzar.


## 2. Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

import xgboost as xgb

pd.set_option('display.max_columns', 50)

PROCESSED_PATH = Path('../data/processed')
MODELS_PATH = Path('../models')
MODELS_PATH.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


## 3. Carga del dataset procesado

In [2]:
dataset = pd.read_parquet(PROCESSED_PATH / 'dataset_localidad_franja_fecha.parquet')
print('Filas:', dataset.shape[0], '| Columnas:', dataset.shape[1])
dataset.head()


Filas: 204560 | Columnas: 16


,Localidad,Franja_Horaria,Fecha_Acc,Num_Accidentes,Dia_Semana,Mes,Es_Fin_de_Semana,Es_Festivo,Accidentes_Prom_7d,Accidentes_Prom_30d,Accidentes_Semana_Anterior,Sin_Historial_Accidentes_Prom_7d,Sin_Historial_Accidentes_Prom_30d,Sin_Historial_Accidentes_Semana_Anterior,Periodo,Alto_Riesgo
0,ANTONIO NARIÑO,Madrugada,2018-01-01,0,Monday,1,0,1,0.0,0.0,0.0,1,1,1,train,0
1,ANTONIO NARIÑO,Madrugada,2018-01-02,0,Tuesday,1,0,0,0.0,0.0,0.0,0,0,1,train,0
2,ANTONIO NARIÑO,Madrugada,2018-01-03,0,Wednesday,1,0,0,0.0,0.0,0.0,0,0,1,train,0
3,ANTONIO NARIÑO,Madrugada,2018-01-04,0,Thursday,1,0,0,0.0,0.0,0.0,0,0,1,train,0
4,ANTONIO NARIÑO,Madrugada,2018-01-05,0,Friday,1,0,0,0.0,0.0,0.0,0,0,1,train,0


## 4. Definición de variables X e y

**Variables excluidas de `X` y por qué:**

| Columna | Motivo de exclusión |
|---|---|
| `Fecha_Acc` | Identificador temporal, no generaliza; ya está representada por `Dia_Semana`, `Mes`, `Es_Fin_de_Semana`, `Es_Festivo` |
| `Num_Accidentes` | Es la misma información usada para construir `Alto_Riesgo` — incluirla sería *data leakage* |
| `Periodo` | Se usa para el split, no como predictora |

Todo lo demás (calendario + históricas + flags de historial insuficiente) sí es válido como predictor.


In [3]:
cat_cols = ['Localidad', 'Franja_Horaria', 'Dia_Semana']

num_cols = [
    'Mes', 'Es_Fin_de_Semana', 'Es_Festivo',
    'Accidentes_Prom_7d', 'Accidentes_Prom_30d', 'Accidentes_Semana_Anterior',
    'Sin_Historial_Accidentes_Prom_7d', 'Sin_Historial_Accidentes_Prom_30d',
    'Sin_Historial_Accidentes_Semana_Anterior'
]

X = dataset[cat_cols + num_cols]
y = dataset['Alto_Riesgo']

print('Variables categóricas:', cat_cols)
print('Variables numéricas:', num_cols)
print('Total de predictoras:', len(cat_cols) + len(num_cols))


Variables categóricas: ['Localidad', 'Franja_Horaria', 'Dia_Semana']
Variables numéricas: ['Mes', 'Es_Fin_de_Semana', 'Es_Festivo', 'Accidentes_Prom_7d', 'Accidentes_Prom_30d', 'Accidentes_Semana_Anterior', 'Sin_Historial_Accidentes_Prom_7d', 'Sin_Historial_Accidentes_Prom_30d', 'Sin_Historial_Accidentes_Semana_Anterior']
Total de predictoras: 12


## 5. Reutilización del split temporal (`Periodo`)

Se reutiliza exactamente el split construido en el Notebook 3 (`train` ≤ 2022-12-31,
`test` = 2023–2024) — no se genera un `train_test_split` nuevo, para no romper la coherencia con
el umbral de `Alto_Riesgo` que ya fue calculado respetando esa misma frontera temporal.


### 5.1 Conjunto de entrenamiento

In [4]:
train_mask = dataset['Periodo'] == 'train'

X_train = X[train_mask]
y_train = y[train_mask]

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)


X_train: (146080, 12)
y_train: (146080,)


### 5.2 Conjunto de prueba

In [5]:
test_mask = dataset['Periodo'] == 'test'

X_test = X[test_mask]
y_test = y[test_mask]

print('X_test:', X_test.shape)
print('y_test:', y_test.shape)


X_test: (58480, 12)
y_test: (58480,)


### 5.3 Distribución de la variable objetivo

In [6]:
resumen_dist = pd.DataFrame({
    'train': y_train.value_counts(normalize=True).sort_index(),
    'test': y_test.value_counts(normalize=True).sort_index()
})
resumen_dist.index = ['No Alto Riesgo (0)', 'Alto Riesgo (1)']
resumen_dist.round(4)


,train,test
No Alto Riesgo (0),0.8207,0.9369
Alto Riesgo (1),0.1793,0.0631


### Interpretación

La proporción de `Alto_Riesgo` es notablemente más baja en test (2023–2024) que en train
(2018–2022). Esto ya se identificó en el Notebook 3 y se relaciona con el menor volumen de
registros de 2023–2024 respecto a años anteriores (posible subregistro), no con una reducción
real de accidentes. Este desbalance entre periodos se debe tener presente al interpretar las
métricas del modelo sobre test en el Notebook 05 — un desempeño más bajo ahí no necesariamente
indica que el modelo generaliza mal, sino que el período de prueba tiene una distribución distinta
al de entrenamiento (*data drift*).


## 6. Preprocesamiento

**Regla general para evitar fuga de información:** todo transformador (encoder, scaler) se ajusta
(`fit`) **solo con `X_train`**, y luego se aplica (`transform`) tanto a train como a test. Ajustar
con el dataset completo filtraría información del período de prueba hacia el entrenamiento.


### 6.1 Codificación de variables categóricas

`Localidad`, `Franja_Horaria` y `Dia_Semana` se codifican con One-Hot Encoding — ninguno de los
tres modelos acepta texto directamente. `handle_unknown='ignore'` evita errores si test tuviera
alguna categoría no vista en train (no debería ocurrir aquí, pero es buena práctica).


In [7]:
encoder_cat = OneHotEncoder(handle_unknown='ignore')
encoder_cat.fit(X_train[cat_cols])

print('Categorías codificadas por variable:')
for col, cats in zip(cat_cols, encoder_cat.categories_):
    print(f'  {col}: {len(cats)} categorías')


Categorías codificadas por variable:
  Localidad: 20 categorías
  Franja_Horaria: 4 categorías
  Dia_Semana: 7 categorías


### 6.2 Escalado de variables numéricas

**Solo necesario para Regresión Logística** — es sensible a la escala de las variables (`Mes` va
de 1 a 12, `Accidentes_Prom_30d` puede ir de 0 a 14). Random Forest y XGBoost son invariantes a
escala, así que para esos dos se usan las variables numéricas sin transformar.


In [8]:
# Preprocesador para Regresión Logística: One-Hot + Escalado
preprocesador_logistica = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols)
])

# Preprocesador para modelos de árboles: solo One-Hot, numéricas sin transformar
preprocesador_arboles = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
], remainder='passthrough')

preprocesador_logistica.fit(X_train)
preprocesador_arboles.fit(X_train)

X_train_log = preprocesador_logistica.transform(X_train)
X_test_log = preprocesador_logistica.transform(X_test)

X_train_arb = preprocesador_arboles.transform(X_train)
X_test_arb = preprocesador_arboles.transform(X_test)

print('Shape tras preprocesamiento (Logística):', X_train_log.shape)
print('Shape tras preprocesamiento (Árboles):', X_train_arb.shape)


Shape tras preprocesamiento (Logística): (146080, 40)
Shape tras preprocesamiento (Árboles): (146080, 40)


### 6.3 Manejo del desbalance de clases

Con ~18% de casos positivos en train, los tres modelos usan su mecanismo nativo de balanceo en
vez de remuestrear los datos (SMOTE, undersampling), para no alterar artificialmente la
distribución temporal ya construida:

- **Regresión Logística / Random Forest:** `class_weight='balanced'` — pondera automáticamente el
  error de la clase minoritaria en proporción inversa a su frecuencia.
- **XGBoost:** `scale_pos_weight` — se calcula manualmente como la razón negativos/positivos en train.


In [9]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight para XGBoost: {scale_pos_weight:.3f}')


scale_pos_weight para XGBoost: 4.576


## 7. Modelo 1: Regresión Logística

Modelo baseline — simple e interpretable, sirve como punto de referencia para evaluar si los
modelos más complejos (Random Forest, XGBoost) realmente aportan una mejora.


In [10]:
modelo_logistica = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE
)

modelo_logistica.fit(X_train_log, y_train)

pred_logistica = modelo_logistica.predict(X_test_log)
proba_logistica = modelo_logistica.predict_proba(X_test_log)[:, 1]

print('Regresión Logística — desempeño en test:')
print(f'  F1-score:  {f1_score(y_test, pred_logistica):.4f}')
print(f'  AUC-ROC:   {roc_auc_score(y_test, proba_logistica):.4f}')
print(f'  Precision: {precision_score(y_test, pred_logistica):.4f}')
print(f'  Recall:    {recall_score(y_test, pred_logistica):.4f}')


Regresión Logística — desempeño en test:
  F1-score:  0.1803
  AUC-ROC:   0.6577
  Precision: 0.1330
  Recall:    0.2798


## 8. Modelo 2: Random Forest

**Nota sobre regularización:** se limita `max_depth=12` y `min_samples_leaf=20`. Sin estas
restricciones, los árboles crecen sin control con este volumen de datos (146,080 filas de train),
generando un modelo de más de 1 GB en disco y con alto riesgo de sobreajuste. Limitar la
profundidad y el tamaño mínimo de hoja reduce el modelo a ~15 MB sin pérdida relevante de
desempeño, y de paso mejora su capacidad de generalización.


In [11]:
modelo_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

modelo_rf.fit(X_train_arb, y_train)

pred_rf = modelo_rf.predict(X_test_arb)
proba_rf = modelo_rf.predict_proba(X_test_arb)[:, 1]

print('Random Forest — desempeño en test:')
print(f'  F1-score:  {f1_score(y_test, pred_rf):.4f}')
print(f'  AUC-ROC:   {roc_auc_score(y_test, proba_rf):.4f}')
print(f'  Precision: {precision_score(y_test, pred_rf):.4f}')
print(f'  Recall:    {recall_score(y_test, pred_rf):.4f}')


Random Forest — desempeño en test:
  F1-score:  0.1551
  AUC-ROC:   0.6506
  Precision: 0.1071
  Recall:    0.2809


## 9. Modelo 3: XGBoost

In [12]:
modelo_xgb = xgb.XGBClassifier(
    n_estimators=200,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric='logloss'
)

modelo_xgb.fit(X_train_arb, y_train)

pred_xgb = modelo_xgb.predict(X_test_arb)
proba_xgb = modelo_xgb.predict_proba(X_test_arb)[:, 1]

print('XGBoost — desempeño en test:')
print(f'  F1-score:  {f1_score(y_test, pred_xgb):.4f}')
print(f'  AUC-ROC:   {roc_auc_score(y_test, proba_xgb):.4f}')
print(f'  Precision: {precision_score(y_test, pred_xgb):.4f}')
print(f'  Recall:    {recall_score(y_test, pred_xgb):.4f}')


XGBoost — desempeño en test:
  F1-score:  0.1973
  AUC-ROC:   0.7053
  Precision: 0.1476
  Recall:    0.2972


## 10. Validación cruzada

**Importante:** la validación cruzada se hace únicamente **dentro de train** — el test se
reserva intacto para la evaluación final del Notebook 05. El objetivo aquí es medir qué tan
estable es cada modelo ante distintas particiones de los datos de entrenamiento, no volver a
tocar el período de prueba.


### 10.1 StratifiedKFold

Se usa `StratifiedKFold` (no `KFold` simple) para que cada partición conserve la misma
proporción de la clase minoritaria (`Alto_Riesgo = 1`) que el conjunto de train completo — con un
desbalance de ~18%, un `KFold` sin estratificar podría dejar algún fold con muy pocos positivos y
volver inestables las métricas.


In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


### 10.2 Métricas de validación

In [14]:
scoring = ['f1', 'roc_auc', 'precision', 'recall']

resultados_cv = {}

resultados_cv['Regresión Logística'] = cross_validate(
    modelo_logistica, X_train_log, y_train, cv=cv, scoring=scoring
)
resultados_cv['Random Forest'] = cross_validate(
    modelo_rf, X_train_arb, y_train, cv=cv, scoring=scoring
)
resultados_cv['XGBoost'] = cross_validate(
    modelo_xgb, X_train_arb, y_train, cv=cv, scoring=scoring
)

resumen_cv = pd.DataFrame({
    modelo: {
        metrica: f"{res[f'test_{metrica}'].mean():.4f} (+/- {res[f'test_{metrica}'].std():.4f})"
        for metrica in scoring
    }
    for modelo, res in resultados_cv.items()
}).T

resumen_cv


,f1,roc_auc,precision,recall
Regresión Logística,0.3807 (+/- 0.0027),0.6838 (+/- 0.0036),0.2669 (+/- 0.0022),0.6633 (+/- 0.0042)
Random Forest,0.3987 (+/- 0.0036),0.7091 (+/- 0.0022),0.2846 (+/- 0.0028),0.6658 (+/- 0.0083)
XGBoost,0.3981 (+/- 0.0029),0.7065 (+/- 0.0024),0.2889 (+/- 0.0024),0.6402 (+/- 0.0078)


### Interpretación

El promedio y la desviación estándar de cada métrica a través de los 5 folds dan una idea de qué
tan estable es cada modelo ante distintas particiones de train — una desviación alta sugiere que
el desempeño depende mucho de qué datos específicos caen en cada fold, lo cual es relevante a
la hora de justificar la elección final del modelo en la tesis.


## 11. Comparación preliminar de modelos

Resumen de las métricas obtenidas sobre el conjunto de **test** (2023–2024) para los tres
modelos. Esta es una comparación preliminar para decidir con qué modelo(s) avanzar — la
evaluación exhaustiva (matriz de confusión, curva ROC, importancia de variables) se hace en
`05_Evaluacion.ipynb`.


In [15]:
comparacion = pd.DataFrame({
    'Regresión Logística': {
        'F1-score': f1_score(y_test, pred_logistica),
        'AUC-ROC': roc_auc_score(y_test, proba_logistica),
        'Precision': precision_score(y_test, pred_logistica),
        'Recall': recall_score(y_test, pred_logistica),
    },
    'Random Forest': {
        'F1-score': f1_score(y_test, pred_rf),
        'AUC-ROC': roc_auc_score(y_test, proba_rf),
        'Precision': precision_score(y_test, pred_rf),
        'Recall': recall_score(y_test, pred_rf),
    },
    'XGBoost': {
        'F1-score': f1_score(y_test, pred_xgb),
        'AUC-ROC': roc_auc_score(y_test, proba_xgb),
        'Precision': precision_score(y_test, pred_xgb),
        'Recall': recall_score(y_test, pred_xgb),
    },
}).T.round(4)

comparacion


,F1-score,AUC-ROC,Precision,Recall
Regresión Logística,0.1803,0.6577,0.1330,0.2798
Random Forest,0.1551,0.6506,0.1071,0.2809
XGBoost,0.1973,0.7053,0.1476,0.2972


### Interpretación

Esta tabla es preliminar — sirve para tener una primera lectura de qué modelo(s) muestran mejor
desempeño relativo antes de pasar al análisis exhaustivo del Notebook 05. Conviene recordar que
las métricas sobre test están afectadas por el *data drift* ya identificado (menor proporción de
`Alto_Riesgo` en 2023–2024), así que la comparación entre modelos es más informativa que el valor
absoluto de cada métrica.


## 12. Guardado de modelos

Se guardan los tres modelos entrenados junto con sus respectivos preprocesadores (necesarios para
transformar cualquier dato nuevo de la misma forma antes de predecir), listos para la evaluación
detallada del Notebook 05.


In [16]:
joblib.dump(modelo_logistica, MODELS_PATH / 'modelo_logistica.pkl')
joblib.dump(preprocesador_logistica, MODELS_PATH / 'preprocesador_logistica.pkl')

joblib.dump(modelo_rf, MODELS_PATH / 'modelo_random_forest.pkl')
joblib.dump(modelo_xgb, MODELS_PATH / 'modelo_xgboost.pkl')
joblib.dump(preprocesador_arboles, MODELS_PATH / 'preprocesador_arboles.pkl')

print('Modelos y preprocesadores guardados en:', MODELS_PATH.resolve())
print(sorted(p.name for p in MODELS_PATH.glob('*.pkl')))


Modelos y preprocesadores guardados en: /Users/camilo/Library/CloudStorage/OneDrive-FundaciónUniversitariaKonradLorenz/Siniestralidad_Vial/models
['modelo_logistica.pkl', 'modelo_random_forest.pkl', 'modelo_xgboost.pkl', 'preprocesador_arboles.pkl', 'preprocesador_logistica.pkl']


## Resumen de decisiones tomadas en esta fase

| Decisión | Justificación |
|---|---|
| Reutilizar `Periodo` del Notebook 3 (no re-splitear) | Mantiene coherencia con el umbral de `Alto_Riesgo`, calculado sin fuga temporal |
| `Fecha_Acc`, `Num_Accidentes`, `Periodo` excluidas de `X` | Evitar *data leakage* y variables no generalizables |
| Encoders/scalers ajustados solo con `X_train` | Evita fuga de información del período de prueba hacia el entrenamiento |
| Escalado solo para Regresión Logística | Random Forest y XGBoost son invariantes a escala |
| `class_weight='balanced'` / `scale_pos_weight` | Maneja el desbalance (~18% positivos) sin alterar la estructura temporal de los datos |
| `StratifiedKFold` en vez de `KFold` | Mantiene la proporción de la clase minoritaria en cada partición |
| Validación cruzada solo dentro de train | El test se reserva intacto para la evaluación final del Notebook 05 |

**Siguiente notebook:** `05_Evaluacion.ipynb` — matriz de confusión, curva ROC, importancia de variables y análisis comparativo detallado sobre los tres modelos guardados aquí.
